# 🌍 NewsPulse — Spark Analysis
**Ryan Adya Purwanto (5027231046)**

Analisis 3 pertanyaan bisnis dari data berita yang tersimpan di HDFS:
1. Kata paling sering muncul di judul berita
2. Distribusi berita per sumber
3. Volume publikasi per jam

In [8]:
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NewsPulse") \
    .getOrCreate()

print("Spark Version:", spark.version)
print("SparkSession siap!")

Spark Version: 4.1.1
SparkSession siap!


In [9]:
# Load RSS
df_rss = spark.read.option("multiLine", True).option("mode", "DROPMALFORMED").json(
    "C:/Users/Ryan/Documents/kelompok7-ets-bigdata/spark/rss.json"
)

# Load API (pakai list)
df_api = spark.read.option("multiLine", True).option("mode", "DROPMALFORMED").json([
    "C:/Users/Ryan/Documents/kelompok7-ets-bigdata/spark/api1.json",
    "C:/Users/Ryan/Documents/kelompok7-ets-bigdata/spark/api2.json",
    "C:/Users/Ryan/Documents/kelompok7-ets-bigdata/spark/api3.json"
])

print(f"RSS records: {df_rss.count()}")
print(f"API records: {df_api.count()}")

df_rss.printSchema()
df_api.printSchema()

RSS records: 50
API records: 12
root
 |-- description: string (nullable = true)
 |-- fetched_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- published_at: string (nullable = true)
 |-- source: string (nullable = true)
 |-- source_api: string (nullable = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)

root
 |-- category: string (nullable = true)
 |-- description: string (nullable = true)
 |-- fetched_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- published_at: string (nullable = true)
 |-- source: string (nullable = true)
 |-- source_api: string (nullable = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)



In [10]:
from pyspark.sql import functions as F

# Pilih kolom yang sama
cols = ["title", "source", "published_at", "fetched_at", "url", "description"]

df_all = df_rss.select(cols).union(df_api.select(cols))

print(f"Total semua berita: {df_all.count()}")
df_all.createOrReplaceTempView("berita")
df_all.show(5, truncate=True)

Total semua berita: 62
+--------------------+------+-------------------+--------------------+--------------------+--------------------+
|               title|source|       published_at|          fetched_at|                 url|         description|
+--------------------+------+-------------------+--------------------+--------------------+--------------------+
|Gibran akan Hadir...| Tempo|2026-05-06T13:51:55|2026-05-06T21:19:...|https://nasional....|Gibran bersedia h...|
|Sekolah Rakyat Do...| Tempo|2026-05-06T12:38:57|2026-05-06T21:19:...|https://nasional....|Program Sekolah R...|
|Abdul Mu'ti Bicar...| Tempo|2026-05-06T11:54:25|2026-05-06T21:19:...|https://nasional....|Keberlangsungan n...|
|Pemerintah Bakal ...| Tempo|2026-05-06T11:43:32|2026-05-06T21:19:...|https://nasional....|setiap sekolah ak...|
|Boraks di MBG, Ke...| Tempo|2026-05-06T11:28:22|2026-05-06T21:19:...|https://nasional....|Kepala BGN memint...|
+--------------------+------+-------------------+--------------------+---

In [11]:
# Ryan Adya Purwanto — Register sebagai SQL View
df_all.createOrReplaceTempView("berita")
print("View 'berita' siap untuk Spark SQL!")

View 'berita' siap untuk Spark SQL!


## Analisis 1 — Kata Paling Sering di Judul Berita
Menggunakan `split()` dan `explode()` untuk memecah judul menjadi kata-kata,
lalu filter stopwords dan hitung frekuensi.

In [12]:
# Ryan Adya Purwanto — Analisis 1: Top Words
stopwords = [
    "dan", "yang", "di", "ke", "dari", "untuk", "dengan",
    "ini", "itu", "pada", "dalam", "adalah", "akan", "tidak",
    "ada", "telah", "juga", "lebih", "sudah", "saat", "bisa",
    "oleh", "atas", "atau", "karena", "jika", "tapi", "bagi",
    "para", "pun", "hal", "nya", "kami", "kita", "mereka",
    "the", "a", "an", "of", "in", "to", "and", "for"
]

result_words = spark.sql(f"""
    SELECT word, COUNT(*) AS count
    FROM (
        SELECT explode(split(lower(title), '[^a-zA-Z0-9äöüÄÖÜ]+')) AS word
        FROM berita
        WHERE title IS NOT NULL
    )
    WHERE word NOT IN ({','.join([f"'{w}'" for w in stopwords])})
      AND length(word) > 3
    GROUP BY word
    ORDER BY count DESC
    LIMIT 20
""")

print("=" * 40)
print("  TOP 20 KATA DI JUDUL BERITA")
print("=" * 40)
result_words.show(20, truncate=False)

print("\n📌 Interpretasi:")
print("Kata-kata di atas mencerminkan isu yang paling banyak dibicarakan media nasional hari ini.")
print("Semakin tinggi frekuensi, semakin banyak artikel yang membahas topik tersebut.")

  TOP 20 KATA DI JUDUL BERITA
+-----------+-----+
|word       |count|
+-----------+-----+
|2026       |7    |
|sekolah    |7    |
|prabowo    |6    |
|daerah     |5    |
|apresiasi  |5    |
|rakyat     |5    |
|kalimantan |4    |
|berprestasi|4    |
|pemerintah |4    |
|pengadaan  |3    |
|pesantren  |3    |
|polri      |3    |
|jadi       |3    |
|soal       |3    |
|kekerasan  |3    |
|bakal      |3    |
|indonesia  |3    |
|guru       |3    |
|regional   |3    |
|sepatu     |3    |
+-----------+-----+


📌 Interpretasi:
Kata-kata di atas mencerminkan isu yang paling banyak dibicarakan media nasional hari ini.
Semakin tinggi frekuensi, semakin banyak artikel yang membahas topik tersebut.


## Analisis 2 — Distribusi Berita per Sumber
Menghitung jumlah artikel dari setiap sumber media menggunakan `groupBy`.

In [13]:
# Ryan Adya Purwanto — Analisis 2: Distribusi per Sumber
result_sources = spark.sql("""
    SELECT
        source,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS persen
    FROM berita
    WHERE source IS NOT NULL
    GROUP BY source
    ORDER BY count DESC
""")

print("=" * 50)
print("  DISTRIBUSI BERITA PER SUMBER")
print("=" * 50)
result_sources.show(truncate=False)

print("\n📌 Interpretasi:")
print("Distribusi ini menunjukkan kontribusi tiap sumber media dalam pipeline.")
print("Ketidakseimbangan bisa terjadi jika satu sumber update lebih sering dari yang lain.")

  DISTRIBUSI BERITA PER SUMBER
+--------------+-----+------+
|source        |count|persen|
+--------------+-----+------+
|Tempo         |50   |80.6  |
|detiksport    |4    |6.5   |
|CNN Indonesia |2    |3.2   |
|detikInet     |1    |1.6   |
|CNBC Indonesia|1    |1.6   |
|investor.id   |1    |1.6   |
|Wolipop       |1    |1.6   |
|detikNews     |1    |1.6   |
|Trans TV      |1    |1.6   |
+--------------+-----+------+


📌 Interpretasi:
Distribusi ini menunjukkan kontribusi tiap sumber media dalam pipeline.
Ketidakseimbangan bisa terjadi jika satu sumber update lebih sering dari yang lain.


## Analisis 3 — Volume Publikasi per Jam
Menghitung berapa banyak berita dipublikasikan di setiap jam menggunakan `HOUR()`.

In [14]:
# Ryan Adya Purwanto — Analisis 3: Volume per Jam
result_hours = spark.sql("""
    SELECT
        HOUR(TO_TIMESTAMP(published_at)) AS hour,
        COUNT(*) AS count
    FROM berita
    WHERE published_at IS NOT NULL
      AND published_at != ''
    GROUP BY hour
    ORDER BY hour
""")

print("=" * 40)
print("  VOLUME BERITA PER JAM")
print("=" * 40)
result_hours.show(24, truncate=False)

# Temukan jam tersibuk
busiest = result_hours.orderBy(F.desc("count")).first()
print(f"\n📌 Interpretasi:")
print(f"Jam tersibuk: {busiest['hour']}:00 WIB dengan {busiest['count']} artikel.")
print("Pola ini membantu PR agency menentukan kapan harus memantau media secara intensif.")

  VOLUME BERITA PER JAM
+----+-----+
|hour|count|
+----+-----+
|0   |2    |
|1   |3    |
|3   |1    |
|4   |4    |
|5   |2    |
|6   |3    |
|7   |5    |
|8   |14   |
|9   |9    |
|10  |5    |
|11  |3    |
|12  |4    |
|13  |3    |
|14  |2    |
|15  |2    |
+----+-----+


📌 Interpretasi:
Jam tersibuk: 8:00 WIB dengan 14 artikel.
Pola ini membantu PR agency menentukan kapan harus memantau media secara intensif.


In [15]:
# Ryan Adya Purwanto — Simpan Hasil ke Dashboard JSON
import json
import datetime

# Kumpulkan hasil
words_list   = [{"word": r["word"], "count": r["count"]} for r in result_words.collect()]
sources_list = [{"source": r["source"], "count": r["count"], "persen": float(r["persen"])} for r in result_sources.collect()]
hours_list   = [{"hour": r["hour"], "count": r["count"]} for r in result_hours.collect() if r["hour"] is not None]

spark_results = {
    "top_words":    words_list,
    "sources":      sources_list,
    "hours":        hours_list,
    "generated_at": datetime.datetime.now().isoformat()
}

# Simpan ke dashboard/data/spark_results.json
output_path = "C:/Users/Ryan/Documents/kelompok7-ets-bigdata/dashboard/data/spark_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(spark_results, f, ensure_ascii=False, indent=2)

print(f"✅ Hasil tersimpan ke: {output_path}")
print(f"   - {len(words_list)} kata trending")
print(f"   - {len(sources_list)} sumber berita")
print(f"   - {len(hours_list)} slot jam")

✅ Hasil tersimpan ke: C:/Users/Ryan/Documents/kelompok7-ets-bigdata/dashboard/data/spark_results.json
   - 20 kata trending
   - 9 sumber berita
   - 15 slot jam


## Ringkasan Analisis

| Analisis | Metode | Insight |
|---|---|---|
| Kata trending | `split() + explode() + filter stopwords` | Topik paling hangat hari ini |
| Distribusi sumber | `groupBy source + count` | Kontribusi tiap media |
| Volume per jam | `HOUR(TO_TIMESTAMP) + groupBy` | Jam paling aktif publikasi berita |

**Jawaban pertanyaan bisnis:**
> Kata-kata dengan frekuensi tertinggi mencerminkan topik paling hangat. 
> Jam tersibuk menunjukkan kapan PR agency harus paling waspada memantau media.